In [ ]:
import pandas as pd
import numpy as np

# Load the data
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

# Display the first few rows of the training data
print("Training Data Head:")
print(train_df.head())

# Display the shape of the dataframes
print(f"\nTraining data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Separate features (pixels) and labels (digits)
X_train_full = train_df.drop(columns=['label'])
y_train_full = train_df['label']

# 2. Visualize some of the digits
plt.figure(figsize=(10, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    # Reshape the 784 pixels into a 28x28 image
    image = X_train_full.iloc[i].values.reshape(28, 28)
    plt.imshow(image, cmap='gray')
    plt.title(f"Label: {y_train_full.iloc[i]}")
    plt.axis('off')
plt.suptitle("Sample Digits from the Training Set", fontsize=16)
plt.show()

# 3. Check the distribution of the labels
plt.figure(figsize=(10, 5))
sns.countplot(x=y_train_full)
plt.title("Distribution of Digit Labels in Training Data")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

In [ ]:
from tensorflow.keras.utils import to_categorical

# 1. Normalize the pixel values to be between 0 and 1
X_train_full = X_train_full / 255.0
# Also normalize the test data, which we will use later
test_images = test_df / 255.0

# 2. Reshape the data for the CNN
# The shape should be (number_of_images, height, width, color_channels)
X_train_full = X_train_full.values.reshape(-1, 28, 28, 1)
test_images = test_images.values.reshape(-1, 28, 28, 1)

# 3. One-hot encode the labels
y_train_full_cat = to_categorical(y_train_full, num_classes=10)

# Print the new shapes to verify
print(f"Training images shape: {X_train_full.shape}")
print(f"Test images shape: {test_images.shape}")
print(f"Categorical training labels shape: {y_train_full_cat.shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the model architecture
model = Sequential([
    # First convolutional block
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),

    # Second convolutional block
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Flatten the feature maps to feed into dense layers
    Flatten(),

    # Dense layer for classification
    Dense(128, activation='relu'),
    
    # Dropout layer to prevent overfitting
    Dropout(0.5),

    # Output layer with 10 units (for 10 digits) and softmax activation
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Display the model's architecture
model.summary()

In [ ]:
from sklearn.model_selection import train_test_split

# We'll use 90% for training and 10% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full_cat, test_size=0.1, random_state=42
)

# Set training parameters
epochs = 15
batch_size = 64

# Train the model
history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val)
)

In [ ]:
import matplotlib.pyplot as plt

# Visualize the training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plotting accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

# Plotting loss
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.show()

# Make predictions on the test data
predictions = model.predict(test_images)

# The model outputs probabilities for each class. We want the class with the highest probability.
predicted_classes = np.argmax(predictions, axis=1)

In [ ]:
# Create the submission file
submission_df = pd.DataFrame({
    'ImageId': range(1, len(predicted_classes) + 1),
    'Label': predicted_classes
})

# Save the submission file
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("Submission file created successfully!")
print(submission_df.head())